# Memory-Based Cognitive Twin
## Personalized Decision Prediction and Email Response Generation using LLMs

**Research Question**: *Can historical memory help an AI predict a user's decision and generate a response that resembles how that user would actually respond?*

---

### Pipeline
```
Historical Emails
      ↓
Create Memory Embeddings  (SentenceTransformer)
      ↓
Retrieve TOP 20 Relevant Historical Memories  (cosine similarity, timestamp < T)
      ↓
ML Model Predicts Decision  (Random Forest + TF-IDF + Embeddings)
      ↓
LLM Generates Personalized Response  (Google Gemini)
      ↓
Evaluate Prediction + Generated Response
```

⚠️ **Data**: Synthetic. Results do not represent real human behavior.  
⚠️ **Mock Mode**: If no `LLM_API_KEY` is set, the LLM step uses a mock response (clearly labelled).

## 0. Setup

In [ ]:
import sys, pathlib
ROOT = pathlib.Path('.').resolve().parent   # adjust if running from notebooks/
sys.path.insert(0, str(ROOT))

# Load environment variables from .env
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / '.env')
    print('✓ .env loaded')
except ImportError:
    print('python-dotenv not installed — set LLM_API_KEY manually if needed')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
%matplotlib inline
print('✓ Imports complete')

## 1. Generate Synthetic Dataset

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, str(ROOT / 'scripts' / 'generate_dataset.py')],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## 2. Load and Inspect Data

In [ ]:
from src.preprocessing import load_raw, chronological_split, describe_split
df = load_raw(ROOT / 'data' / 'raw' / 'emails.csv')
print(f'Total records: {len(df)}')
df.head(3)

In [ ]:
print('Decision distribution:')
print(df['decision'].value_counts())
print('\nUsers:', df['user_id'].unique())

## 3. Chronological Split

**Leakage Prevention**: The split is strictly chronological. No test/val timestamps appear in training.

In [ ]:
train_df, val_df, test_df = chronological_split(df)
describe_split(train_df, val_df, test_df)

print(f'\nLeakage check:')
print(f'  Max train timestamp : {train_df["timestamp"].max()}')
print(f'  Min val timestamp   : {val_df["timestamp"].min()}')
print(f'  Min test timestamp  : {test_df["timestamp"].min()}')

## 4. Feature Extraction

In [ ]:
from src.features import fit_tfidf, build_features, make_label_encoder

le = make_label_encoder()
y_train = le.transform(train_df['decision'].values)
y_val   = le.transform(val_df['decision'].values)
y_test  = le.transform(test_df['decision'].values)

# TF-IDF fitted on TRAIN only
tfidf_vec = fit_tfidf(train_df['incoming_email'].tolist())

# Baseline: TF-IDF only
X_train_base, _ = build_features(train_df, tfidf_vec, use_embeddings=False)
X_val_base,   _ = build_features(val_df,   tfidf_vec, use_embeddings=False)
X_test_base,  _ = build_features(test_df,  tfidf_vec, use_embeddings=False)

# CognitiveTwin: TF-IDF + sentence embeddings
X_train_ct, _ = build_features(train_df, tfidf_vec, use_embeddings=True)
X_val_ct,   _ = build_features(val_df,   tfidf_vec, use_embeddings=True)
X_test_ct,  _ = build_features(test_df,  tfidf_vec, use_embeddings=True)

print(f'Baseline feature shape  : {X_train_base.shape}')
print(f'CognitiveTwin feat shape: {X_train_ct.shape}')

## 5. Train ML Models

In [ ]:
from src.models import build_baseline, build_cognitive_twin, train_model, predict

baseline = build_baseline()
ct_model = build_cognitive_twin()

train_model(baseline, X_train_base, y_train)
train_model(ct_model, X_train_ct,   y_train)
print('✓ Both models trained')

## 6. Evaluate Decision Prediction

In [ ]:
from src.evaluation import evaluate_predictions

y_pred_base = le.inverse_transform(predict(baseline, X_test_base))
y_pred_ct   = le.inverse_transform(predict(ct_model, X_test_ct))
y_test_str  = le.inverse_transform(y_test)

base_metrics = evaluate_predictions(y_test_str, y_pred_base, 'Baseline (LogReg+TF-IDF)')
ct_metrics   = evaluate_predictions(y_test_str, y_pred_ct,   'CognitiveTwin (RF+Embeddings)')

### Figure 1: Prediction Performance Comparison

In [ ]:
from src.evaluation import plot_prediction_comparison, plot_confusion_matrix
plot_prediction_comparison(base_metrics, ct_metrics)
from IPython.display import Image
Image(str(ROOT / 'results' / 'figures' / 'fig1_prediction_comparison.png'))

### Figure 2: Confusion Matrix

In [ ]:
plot_confusion_matrix(y_test_str, y_pred_ct)
Image(str(ROOT / 'results' / 'figures' / 'fig2_confusion_matrix.png'))

## 7. Build Memory Store

In [ ]:
from src.memory import embed_texts, build_memory_store, retrieve_memories

all_embs = embed_texts(df['incoming_email'].tolist())
memory_store = build_memory_store(df, all_embs)
print(f'✓ Memory store ready — {len(df)} records')

## 8. Memory Retrieval Demo

Retrieve TOP-20 memories for a sample test email.

In [ ]:
# Pick a test email
row = test_df.iloc[0]
global_pos = df[df['record_id'] == row['record_id']].index[0]
target_emb = all_embs[global_pos]

memories = retrieve_memories(
    target_record_id  = row['record_id'],
    target_user_id    = row['user_id'],
    target_timestamp  = row['timestamp'],
    target_embedding  = target_emb,
    memory_store      = memory_store,
    top_k             = 20,
)

print(f'Target email  : {row["email_subject"]}  ({row["user_id"]})')
print(f'Target time   : {row["timestamp"]}')
print(f'Memories found: {len(memories)}')
print(f'All before T  : {all(pd.Timestamp(m["timestamp"]) < row["timestamp"] for m in memories)}')
print()
for m in memories[:5]:
    print(f'  [{m["timestamp"]}] sim={m["similarity_score"]:.3f}  {m["decision"]}  |  {m["email_subject"]}')

### Figure 3: Memory Similarity Distribution

In [ ]:
from src.evaluation import plot_memory_similarity_distribution

# Collect memories for all test emails
all_mem_lists = []
for _, r in test_df.iterrows():
    gp = df[df['record_id'] == r['record_id']].index[0]
    mems = retrieve_memories(
        r['record_id'], r['user_id'], r['timestamp'], all_embs[gp], memory_store, top_k=20
    )
    if mems:
        all_mem_lists.append(mems)

plot_memory_similarity_distribution(all_mem_lists)
Image(str(ROOT / 'results' / 'figures' / 'fig3_memory_similarity_distribution.png'))

## 9. LLM Response Generation

In [ ]:
from src.response_generator import generate_response, MOCK_MODE

if MOCK_MODE:
    print('⚠  MOCK MODE — set LLM_API_KEY in .env for real Gemini responses')

# Use the first test row
row = test_df.iloc[0]
gp = df[df['record_id'] == row['record_id']].index[0]
target_emb = all_embs[gp]

test_pos = 0
pred_decision = le.inverse_transform(predict(ct_model, X_test_ct[test_pos:test_pos+1]))[0]

mems = retrieve_memories(
    row['record_id'], row['user_id'], row['timestamp'], target_emb, memory_store, top_k=20
)

gen_result = generate_response(
    incoming_email     = row['incoming_email'],
    predicted_decision = pred_decision,
    memories           = mems,
)

print(f'Mode   : {gen_result["mode"]}')
print(f'Model  : {gen_result["model"]}')
print(f'Memories used: {gen_result["n_memories"]}')
print()
print('=== INCOMING EMAIL ===')
print(row['incoming_email'])
print()
print(f'=== ML PREDICTED DECISION: {pred_decision} ===')
print()
print('=== GENERATED RESPONSE ===')
print(gen_result['response'])
print()
print('=== ACTUAL HELD-OUT RESPONSE ===')
print(row['user_response'])

## 10. Response Evaluation

In [ ]:
from src.evaluation import evaluate_response, plot_response_metrics, plot_style_comparison_per_user

# Evaluate on N_DEMO test samples
N_DEMO = 10
response_evals = []
style_data = []

demo_rows = []
for uid in test_df['user_id'].unique():
    uid_rows = test_df[test_df['user_id'] == uid].head(2)
    demo_rows.extend(uid_rows.to_dict('records'))
demo_rows = demo_rows[:N_DEMO]

for r in demo_rows:
    r = pd.Series(r)
    gp = df[df['record_id'] == r['record_id']].index[0]
    t_emb = all_embs[gp]
    mems = retrieve_memories(r['record_id'], r['user_id'], r['timestamp'], t_emb, memory_store)
    
    # Get ML prediction
    row_df = pd.DataFrame([r.to_dict()])
    X_row, _ = build_features(row_df, tfidf_vec, use_embeddings=True)
    pred = le.inverse_transform(predict(ct_model, X_row))[0]

    gen = generate_response(r['incoming_email'], pred, mems)
    ev  = evaluate_response(gen['response'], r['user_response'], r['incoming_email'], pred)
    response_evals.append(ev)
    style_data.append({'user_id': r['user_id'], 'generated': gen['response'], 'actual': r['user_response']})

import numpy as np
agg = {k: round(np.mean([e[k] for e in response_evals]), 4) for k in response_evals[0]}
print('Aggregate response evaluation metrics:')
for k, v in agg.items():
    print(f'  {k:<30}: {v:.4f}')

### Figure 4: Response Evaluation Metrics

In [ ]:
plot_response_metrics(response_evals)
Image(str(ROOT / 'results' / 'figures' / 'fig4_response_metrics.png'))

### Figure 5: Style Comparison per User

In [ ]:
plot_style_comparison_per_user(style_data)
Image(str(ROOT / 'results' / 'figures' / 'fig5_style_comparison.png'))

## 11. Full Demo — Single Email Walkthrough

In [ ]:
# Pick Alice's first test email
alice_test = test_df[test_df['user_id'] == 'alice'].iloc[0]
gp = df[df['record_id'] == alice_test['record_id']].index[0]
t_emb = all_embs[gp]

mems = retrieve_memories(alice_test['record_id'], alice_test['user_id'],
                         alice_test['timestamp'], t_emb, memory_store, top_k=20)

alice_df = pd.DataFrame([alice_test.to_dict()])
X_alice, _ = build_features(alice_df, tfidf_vec, use_embeddings=True)
pred = le.inverse_transform(predict(ct_model, X_alice))[0]

gen = generate_response(alice_test['incoming_email'], pred, mems)
ev  = evaluate_response(gen['response'], alice_test['user_response'],
                         alice_test['incoming_email'], pred)

DIV = '─' * 60
print(DIV)
print('INPUT EMAIL')
print(DIV)
print(alice_test['incoming_email'])

print(f'\n{DIV}')
print(f'ML PREDICTED DECISION: {pred}  (actual: {alice_test["decision"]})')

print(f'\n{DIV}')
print(f'TOP {len(mems)} RETRIEVED MEMORIES')
print(DIV)
for i, m in enumerate(mems[:5], 1):
    print(f'Memory {i}: [{m["timestamp"]}] sim={m["similarity_score"]:.3f} | {m["decision"]} | {m["email_subject"]}')
print(f'... ({len(mems)} total memories)')

print(f'\n{DIV}')
print(f'GENERATED RESPONSE [{gen["mode"].upper()}]')
print(DIV)
print(gen['response'])

print(f'\n{DIV}')
print('ACTUAL HELD-OUT RESPONSE')
print(DIV)
print(alice_test['user_response'])

print(f'\n{DIV}')
print('EVALUATION')
print(DIV)
for k, v in ev.items():
    print(f'  {k:<30}: {v:.4f}')

## Summary

| Metric | Value |
|--------|-------|
| Baseline Accuracy | — |
| CognitiveTwin Accuracy | — |
| Avg Semantic Similarity | — |
| Avg Style Similarity | — |
| Avg Relevance | — |
| Avg Decision Consistency | — |

> Run the cells above to populate the table with real values.

---

### Limitations

- Synthetic data only — results do not generalize to real users.
- Evaluation metrics are automated proxies, not human judgements.
- Small dataset (300 records) limits statistical significance.
- LLM responses in mock mode are placeholders — use real API key for final demo.